In [15]:
#this code will process our files and generate a new one with just the info we need.

#opens .nc file, selects variables, filters and clips to aoi, saves smaller geojson file.

to do: making filtering configurable, so can filter on and off and can make as a func and call w yes/no, if not filter - only save lat and lon

In [36]:
#imports!

from pathlib import Path
import zipfile
import numpy as np
import xarray as xr
import geopandas as gpd
from google.colab import drive

In [37]:
#Configurables!

zip_file = Path("/content/drive/MyDrive/wetransfer_swot_l2_hr_pixc_031_029_234r_20250408t044534_20250408t044545_pic2_01-nc_2026-07-14_0846.zip")
aoi_file = Path("/content/Orco_shapefile_uav.shp")
data_download_path = Path("/content/PIXC_input")
data_processed_path = Path("/content/Orco_processed")
pixc_filename = ("SWOT_L2_HR_PIXC_031_029_234R_20250408T044534_20250408T044545_PGD0_01.nc")

#filters
min_water_class = 3
max_water_class = 5

#naming for output
output_suffix = "_processed.geojson"

In [43]:
import xarray as xr

pixc_filename = "/content/Orco/PIXC_input/SWOT_L2_HR_PIXC_031_029_234R_20250408T044534_20250408T044545_PGD0_01.nc"

ds_PIXC = xr.open_dataset(pixc_filename,group="pixel_cloud")

In [44]:
ds_PIXC["dheight_dphase"]

<xarray.DataArray 'dheight_dphase' (points: 6409635)> Size: 26MB
[6409635 values with dtype=float32]
Coordinates:
    latitude   (points) float64 51MB ...
    longitude  (points) float64 51MB ...
Dimensions without coordinates: points
Attributes:
    long_name:     sensitivity of height estimate to interferogram phase
    units:         m/radian
    quality_flag:  geolocation_qual
    valid_min:     -999999.0
    valid_max:     999999.0
    comment:       Sensitivity of the height estimate to the interferogram ph...

In [18]:
#paths

output_file = (data_processed_path/f"{pixc_file.stem}{output_suffix}")

In [19]:


#okkkk uploading bounding box in the same way Lucas did, no change here rlly.
aoi_shapefile=gpd.read_file(aoi_file).to_crs('EPSG:4326')

#omg right the issue is that hes using the login and we're using only whats uploaded, rightt
#will have to ask charlie, cus maybe then we can just use the server

data_processed_path.mkdir(exist_ok=True, parents=True)





In [39]:

drive.mount("/content/drive")

data_download_path.mkdir(exist_ok=True,parents=True)

#this is the only bit i added rlly, just reading in and extracting from the zip.
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(data_download_path)


#this is basically me saying in my unzipped folder pls print the name of any folders ending
for file in data_download_path.iterdir():
    print(file.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SWOT_L2_HR_PIXC_031_029_234R_20250408T044534_20250408T044545_PIC2_01.nc
SWOT_L2_HR_PIXC_031_029_234R_20250408T044534_20250408T044545_PGD0_01.nc


In [21]:
filename = data_download_path / pixc_filename
ds_PIXC = xr.open_dataset(filename,group="pixel_cloud",engine="h5netcdf")



In [22]:
#getting interferogram cus will have complex and real parts!
interferogram = ds_PIXC["interferogram"].values
print("Interferogram shape:", interferogram.shape)

#splitting it up!!!
interferogram_real = interferogram[:, 0]
interferogram_imag = interferogram[:, 1]

#and reconstructing
complex_interferogram = (interferogram_real+ 1j * interferogram_imag)
wrapped_phase = np.angle(complex_interferogram)

Interferogram shape: (6409635, 2)


In [23]:
#same as always, just making gdf in the same way he did and by adapting what data
#for example, will ad interferogram etc ohase info (maybe convert to phase etc), sep by complex part?

#this is the exact same as lucas's code j with variables added
gdf = gpd.GeoDataFrame(
    data={
        "height": ds_PIXC.height.values.astype("float32"),
        "classification": ds_PIXC.classification.values.astype("uint8"),
        "classification_qual": ds_PIXC.classification_qual.values,
        "bright_land_flag": ds_PIXC.bright_land_flag.values,
        "geolocation_qual": ds_PIXC.geolocation_qual.values,
        "interferogram_qual": ds_PIXC.interferogram_qual.values,
        "sig0": ds_PIXC.sig0.values.astype("float32"),
        "sig0_qual": ds_PIXC.sig0_qual.values,
        "coherent_power": ds_PIXC.coherent_power.values.astype("float32"),
        "latitude": ds_PIXC.latitude.values.astype("float32"),
        "longitude": ds_PIXC.longitude.values.astype("float32"),
        "pixel_area": ds_PIXC.pixel_area.values.astype("float32"),
        "geoid": ds_PIXC.geoid.values.astype("float32"),
        "cross_track": ds_PIXC.cross_track.values.astype("float32"),
        "range_index": ds_PIXC.range_index.values.astype("float32"),
        "azimuth_index": ds_PIXC.azimuth_index.values.astype("float32"),
        "water_frac": ds_PIXC.water_frac.values.astype("float32"),
        "phase_noise_std": ds_PIXC.phase_noise_std.values.astype("float32"),
        "phase_unwrapping_region":ds_PIXC.phase_unwrapping_region.values.astype("float32"),
        "instrument_phase_cor":ds_PIXC.instrument_phase_cor.values.astype("float32"),
        "dlatitude_dphase":ds_PIXC.dlatitude_dphase.values.astype("float32"),
        "dlongitude_dphase":ds_PIXC.dlongitude_dphase.values.astype("float32"),
        "dheight_dphase":ds_PIXC.dheight_dphase.values.astype("float32"),
        "dheight_drange":ds_PIXC.dheight_drange.values.astype("float32"),
        #adding interferogram infooo

        "interferogram_real":interferogram_real.astype("float32"),
        "interferogram_imag":interferogram_imag.astype("float32"),
        "wrapped_phase":wrapped_phase.astype("float32"),

    },



    geometry=gpd.points_from_xy( #same as always here! same w lucas's code
        ds_PIXC.longitude.values,
        ds_PIXC.latitude.values
    )
)

gdf.crs = "EPSG:4326" #setting same coords

print(gdf)
print(gdf.columns) #these are j classic checks to make sure each step is ok...
print("Number of PIXC points:", len(gdf))

              height  classification  classification_qual  bright_land_flag  \
0        2420.751953               1                  3.0               0.0   
1        2420.002441               1                  3.0               0.0   
2        2419.295166               1               2051.0               0.0   
3        2418.624756               1               2051.0               0.0   
4        2417.884033               1               2051.0               0.0   
...              ...             ...                  ...               ...   
6409630   326.592316               1                  2.0               0.0   
6409631   323.998016               1                  0.0               0.0   
6409632   324.323242               1                  2.0               0.0   
6409633   325.154297               1                  2.0               0.0   
6409634   329.538513               1                  2.0               0.0   

         geolocation_qual  interferogram_qual      

In [31]:


#only watery pixels but more strict version to get good measurements:
gdf_processed=gdf[(gdf['classification']>2) & (gdf['geolocation_qual']<4) & (gdf['classification']<6) & (gdf['sig0_qual']==0) & (gdf['geolocation_qual']==0) &\
                      (gdf['classification_qual']==0) & (gdf['interferogram_qual']==0) & (gdf['bright_land_flag']==0) ]

#clipping in the same way! and resaving the independent copy (so doesnt affect previous)

gdf_processed=gpd.clip(gdf_processed,aoi_shapefile)


print("Original PIXC points:", len(gdf))
print("Water pixels:", len(gdf[(gdf["classification"] >= min_water_class)&(gdf["classification"] <= max_water_class)])) #j showing the number of pixels that are satisfying, good info to have?
print("Pixels inside aoi:", len(gdf_processed))

Original PIXC points: 6409635
Water pixels: 379579
Pixels inside aoi: 46


In [32]:
#doing the same, as in if the pixels actually survive the cuts and we have data THEN we move on



if len(gdf_processed) > 0:

    gdf_processed.to_file(output_filename, driver="GeoJSON")
    print("Processed file saved to:")
    print(output_filename)

else:
    print("No PIXC points remained after filtering and clipping.")

Processed file saved to:
/content/Orco_processed/SWOT_L2_HR_PIXC_031_029_234R_20250408T044534_20250408T044545_PGD0_01_processed.geojson
